# Поиск аномалий: выявление нетипичных наблюдений

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Поиск аномалий».

## Что делает метод

Поиск аномалий отвечает на вопрос «что в моих данных выглядит необычно». Алгоритм сначала по большинству данных формирует представление о норме, а затем отмечает наблюдения, которые в эту норму не вписываются. Заранее размечать аномалии не требуется — их обычно слишком мало и они разнообразны.

Типичные научные задачи: контроль качества измерений, поиск сбоев оборудования, выявление редких физических событий, нетипичных пациентов или подозрительных записей. В этом блокноте мы применим метод Isolation Forest, оценим качество и разберём, как читать степень аномальности. Расчётное время прохождения — около пяти минут.

## Интуиция за методом

Представьте опытного лаборанта, который каждый день просматривает сотни однотипных измерений. Он не может объяснить словами все правила «нормы», но сразу замечает запись, которая выбивается из общей картины: не потому что знает, каким должен быть результат, а потому что эта запись не похожа на все остальные.

Isolation Forest работает по схожему принципу: он многократно случайно «разрезает» пространство признаков и замечает, что аномальные точки отделяются очень быстро — они одиноки и поэтому легко изолируются. Нормальные точки, окружённые соседями, изолировать труднее: требуется много разрезов.

**Ключевые понятия, которые встретятся в блокноте:**

- **Признак** — один измеренный параметр наблюдения (столбец таблицы).
- **Аномалия (выброс)** — наблюдение, заметно отличающееся от основного массива данных; может быть ошибкой измерения, редким событием или сбоем.
- **Степень аномальности** — непрерывная числовая оценка того, насколько наблюдение отличается от нормы; чем выше, тем подозрительнее точка.
- **Порог отсечения** — значение степени аномальности, выше которого наблюдение считается аномалией; задаётся параметром `contamination` (ожидаемая доля аномалий).
- **Стандартизация** — приведение признаков к единому масштабу; без неё алгоритм будет реагировать прежде всего на признаки с большим числовым разбросом.
- **AUC-PR** — площадь под кривой точность–полнота; более надёжная метрика, чем обычная точность, когда аномалий мало.

## 1. Установка библиотек

In [ ]:
%pip install -q scikit-learn numpy pandas matplotlib

## 2. Единый стиль графиков

Графики оформляются в едином визуальном стиле сайта «ИИ для учёных». Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py`.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",
    "node_fill":  "#eef1f6",
    "node_text":  "#0e1726",
    "edge":       "#4d5e78",
    "grid":       "#dce2ec",
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Сформируем синтетический набор из двух компонентов:
- **Норма** (480 наблюдений): два компактных облака — модель «нормального режима» с двумя типичными состояниями.
- **Аномалии** (20 наблюдений): равномерно рассеяны по широкой области — модель редких нетипичных событий.

Истинные метки нормы/аномалии нужны лишь в конце для оценки качества; сам алгоритм их не использует. Ячейка ниже создаёт набор данных и показывает первые строки.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_blobs

rng = np.random.default_rng(42)

# Норма: два компактных режима.
X_normal, _ = make_blobs(n_samples=480, centers=[[-4, 0], [4, 1]],
                         cluster_std=1.0, random_state=42)
# Аномалии: равномерно рассеяны по широкой области.
X_anomaly = rng.uniform(low=-12, high=12, size=(20, 2))

X = np.vstack([X_normal, X_anomaly])
true_label = np.r_[np.zeros(len(X_normal)), np.ones(len(X_anomaly))]  # 1 = аномалия
df = pd.DataFrame(X, columns=["признак_1", "признак_2"])

print(f"Всего наблюдений: {len(df)}, из них аномалий: {int(true_label.sum())}")
df.head()

## 4. Применение метода

**Шаг 4.1. Стандартизация признаков.**

Как и в кластеризации, признаки масштабируем перед обучением: без этого алгоритм будет реагировать прежде всего на признак с наибольшим числовым разбросом.

**Шаг 4.2. Обучение Isolation Forest.**

Isolation Forest строит множество случайных деревьев разбиения. Ключевая идея: аномальные точки отделяются от остальных **за меньшее число разбиений** — они одиноки в пространстве признаков. Нормальные точки, плотно окружённые соседями, изолировать труднее.

Параметр `contamination=0.04` — ожидаемая доля аномалий (4 %): используется для установки порога между «нормой» и «аномалией». Результат `fit_predict`: -1 — аномалия, +1 — норма.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

X_scaled = StandardScaler().fit_transform(df)

model = IsolationForest(n_estimators=200, contamination=0.04,
                        random_state=42)
pred = model.fit_predict(X_scaled)            # -1 = аномалия, 1 = норма
is_anomaly = (pred == -1).astype(int)
# Чем меньше score, тем аномальнее наблюдение.
anomaly_score = -model.score_samples(X_scaled)

print(f"Помечено аномалиями: {int(is_anomaly.sum())} наблюдений")

**Шаг 4.3. Оценка качества по известным меткам.**

Ячейка ниже сравнивает предсказания алгоритма с истинными метками (которые мы создали при генерации данных). В реальных задачах этого шага часто нет — разметки аномалий не существует. Здесь он нужен, чтобы убедиться, что алгоритм работает корректно.

**AUC-PR** (площадь под кривой точность–полнота) — надёжная метрика при сильном дисбалансе классов (аномалий мало): значение 1.0 — идеально, 0.0 — хуже случайного угадывания.

In [ ]:
from sklearn.metrics import classification_report, average_precision_score

print(classification_report(true_label, is_anomaly,
                            target_names=["норма", "аномалия"]))
ap = average_precision_score(true_label, anomaly_score)
print(f"Средняя точность ранжирования (AUC-PR): {ap:.3f}")

### Шаг 4.4. Визуализация результата

Ячейка ниже строит два графика:

1. **Карта наблюдений**: аномалии выделены крестами на фоне нормальных точек — видно, что они рассеяны по периферии.
2. **Распределение степени аномальности**: гистограмма показывает, насколько алгоритм «уверен» в аномальности каждого наблюдения. Пунктир — порог отсечения.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2))

# Карта наблюдений
normal_mask = is_anomaly == 0
axes[0].scatter(df.loc[normal_mask, "признак_1"],
                df.loc[normal_mask, "признак_2"],
                s=22, alpha=0.6, color=VIZ["series"][0], label="Норма")
axes[0].scatter(df.loc[~normal_mask, "признак_1"],
                df.loc[~normal_mask, "признак_2"],
                s=70, color=VIZ["series"][2], marker="X",
                label="Аномалия")
axes[0].set_title("Наблюдения и выявленные аномалии")
axes[0].set_xlabel("Признак 1")
axes[0].set_ylabel("Признак 2")
axes[0].legend(loc="best")

# Распределение степени аномальности
axes[1].hist(anomaly_score, bins=40, color=VIZ["series"][1],
             edgecolor=VIZ["background"])
threshold = np.quantile(anomaly_score, 1 - 0.04)
axes[1].axvline(threshold, color=VIZ["series"][2], linestyle="--",
                label="Порог отсечения")
axes[1].set_title("Распределение степени аномальности")
axes[1].set_xlabel("Степень аномальности")
axes[1].set_ylabel("Число наблюдений")
axes[1].legend(loc="upper right")

fig.tight_layout()
plt.show()

**Как читать эти графики:**

- **Карта наблюдений (левый)**: синие круги — нормальные наблюдения, оранжевые кресты — помеченные аномалиями. Аномалии должны находиться на периферии облаков нормальных точек. Если кресты попали внутрь плотного облака — это либо ложное срабатывание (параметр `contamination` завышен), либо реальная «скрытая» аномалия, неотличимая по этим двум признакам.

- **Распределение степени аномальности (правый)**: большинство наблюдений сосредоточено слева (низкая аномальность — норма). Правый хвост — самые подозрительные точки. Пунктирная линия — порог: всё правее неё помечается как аномалия. Сдвиг порога вправо уменьшит число срабатываний, но увеличит число пропущенных аномалий.

## 5. Интерпретация результата

- **Степень аномальности** — непрерывная величина: чем она выше, тем сильнее наблюдение отклоняется от нормы. Удобнее работать не с жёсткой меткой, а с ранжированием — рассматривать наблюдения в порядке убывания этой величины.
- **Порог отсечения** задаётся параметром `contamination`. Его выбирают исходя из допустимой доли ложных срабатываний: ниже порог — больше выявленных аномалий, но и больше ложных тревог.
- **Отчёт классификации и AUC-PR** показывают качество при наличии разметки для проверки; при сильном дисбалансе классов AUC-PR информативнее доли верных ответов.
- **Карта наблюдений**: помеченные точки на периферии облака — ожидаемый результат; аномалия внутри плотной области требует отдельной проверки.

Помните: статистическая аномальность не равна содержательной значимости — каждое выявленное наблюдение следует интерпретировать предметно.

## 5б. Наглядный эксперимент: тепловая карта степени аномальности

Нарисуем «тепловую карту» степени аномальности по всей плоскости признаков: тёмные зоны — там модель считает точки нормальными, светлые — там точки были бы аномалиями. Это позволяет увидеть, как именно Isolation Forest «видит» норму и где она заканчивается.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Сетка для тепловой карты (в стандартизованных координатах).
x_min, x_max = X_scaled[:, 0].min() - 0.5, X_scaled[:, 0].max() + 0.5
y_min, y_max = X_scaled[:, 1].min() - 0.5, X_scaled[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 250),
                     np.linspace(y_min, y_max, 250))

# Степень аномальности по всей плоскости.
grid_scores = -model.score_samples(np.c_[xx.ravel(), yy.ravel()])
grid_scores = grid_scores.reshape(xx.shape)

fig, ax = plt.subplots(figsize=(8, 6))
cf = ax.contourf(xx, yy, grid_scores, levels=30, cmap="YlOrRd", alpha=0.85)
cb = fig.colorbar(cf, ax=ax, label="Степень аномальности")

# Наблюдения поверх тепловой карты.
normal_mask = is_anomaly == 0
ax.scatter(X_scaled[normal_mask, 0], X_scaled[normal_mask, 1],
           s=18, alpha=0.5, color=VIZ["series"][0], label="Норма")
ax.scatter(X_scaled[~normal_mask, 0], X_scaled[~normal_mask, 1],
           s=80, color=VIZ["series"][2], marker="X",
           edgecolors="white", linewidths=0.8, label="Аномалия", zorder=5)

ax.set_title("Тепловая карта степени аномальности (Isolation Forest)")
ax.set_xlabel("Признак 1 (стандартизованный)")
ax.set_ylabel("Признак 2 (стандартизованный)")
ax.legend(loc="upper right")
fig.tight_layout()
plt.show()

**Как читать этот график:**

Цвет фона показывает степень аномальности каждой точки пространства: светло-жёлтый — «нормальная» зона, тёмно-красный — «аномальная». Видно, что модель строит «ореол нормы» вокруг двух плотных облаков: ближе к центру облаков — тем менее подозрительно. На периферии и вдали от облаков степень аномальности резко растёт.

Обратите внимание: между двумя облаками нормы тоже повышенная аномальность — там нет обучающих точек. Это важно учитывать: точка в «пустой» части пространства будет помечена как аномалия, даже если она «между» двумя нормальными режимами.

## 6. Попробуйте на своих данных

Замените демонстрационный набор своими данными. Подготовьте таблицу числовых признаков; разметка аномалий не нужна — метод работает без неё.

1. Загрузите файл в Colab: панель слева, вкладка «Файлы», кнопка загрузки.
2. Снимите комментарии в ячейке ниже, укажите имя файла и при необходимости скорректируйте `contamination`.
3. Последовательно выполните ячейки разделов 4 и 5. Шаги с разметкой (`classification_report`, AUC-PR) выполняйте только при наличии истинных меток.

## Поэкспериментируйте сами

На основном примере попробуйте изменить параметры и перезапустить ячейки раздела 4:

| Параметр | Что изменить | Что наблюдать |
|---|---|---|
| `contamination` | Попробуйте 0.01 и 0.15 | Как меняется число помеченных аномалий? Появляются ли ложные тревоги внутри облаков? |
| `n_estimators` | Уменьшите до 10 | Стабилен ли результат при каждом запуске? |
| Число аномалий | Измените `size=(50, 2)` при создании `X_anomaly` | Что происходит с AUC-PR при большей доле аномалий? |

На тепловой карте (раздел 5б) посмотрите, как меняется форма «ореола нормы» при `contamination=0.20`.

## Краткие выводы

- Isolation Forest выявляет аномалии без разметки: он строит представление о норме по основному массиву данных и помечает то, что плохо в неё вписывается.
- Степень аномальности — непрерывная величина; работайте с ней как с ранжированием, а не только как с жёсткой меткой.
- Параметр `contamination` задаёт ожидаемую долю аномалий и влияет на порог: при неизвестной доле начните с 0.05 и скорректируйте по содержательным критериям.
- Стандартизация обязательна: без неё алгоритм реагирует на признак с наибольшим разбросом.
- Статистическая аномальность не равна содержательной значимости — каждое выявленное наблюдение нужно интерпретировать предметно.
- При наличии разметки аномалий оценивайте качество по AUC-PR, а не по доле верных ответов.

## Три вопроса для самопроверки

Ответьте до того, как раскроете подсказку, — это проверяет, что метод понят, а не просто запущен.

1. На тепловой карте между двумя плотными облаками нормальных точек видна зона повышенной аномальности, хотя там нет ни одного «аномального» примера. Что это говорит об ограничениях Isolation Forest при работе с данными, имеющими несколько нормальных режимов?
2. Вы применили Isolation Forest к своим данным без стандартизации и получили результат. Один из признаков — давление в паскалях (значения ~100 000), другой — температура в кельвинах (~300). Как это повлияет на работу алгоритма?
3. AUC-PR на демонстрационных данных составил ~0.97, тогда как обычная доля верных ответов (accuracy) — ~0.99. Почему при поиске аномалий принято ориентироваться именно на AUC-PR, а не на accuracy?

<details>
<summary>Показать ориентиры для ответов</summary>

1. Isolation Forest строит представление о норме по плотности точек. «Пустая» область между двумя нормальными режимами воспринимается как потенциально аномальная, потому что там мало обучающих примеров. Это фундаментальное ограничение: если нормальные данные имеют несколько разобщённых кластеров, алгоритм будет избыточно помечать межкластерное пространство. Решение — обучать отдельные модели для каждого режима или использовать алгоритмы с явным моделированием смесей.
2. Без стандартизации признак с большим числовым диапазоном (давление) будет доминировать при случайных разбиениях пространства: алгоритм будет разбивать почти всегда по оси давления, игнорируя температуру. Аномалии по температуре окажутся незамечены. Стандартизация обязательна.
3. При сильном дисбалансе классов (аномалий 4 %) даже «тупая» модель, никогда не предсказывающая аномалий, даёт accuracy = 0.96. AUC-PR измеряет качество именно по аномальному классу: насколько точно и полно модель его находит. Высокая accuracy при низком AUC-PR — признак того, что модель практически игнорирует аномалии.
</details>


In [ ]:
# --- Шаблон загрузки своих данных ---
# df = pd.read_csv("my_data.csv")               # ваш файл
# df = df.select_dtypes(include="number")       # оставляем числовые признаки
#
# print(f"Наблюдений: {df.shape[0]}, признаков: {df.shape[1]}")
# Далее повторите ячейки раздела 4 (true_label задайте, если разметка есть).